In [2]:
import arviz as az
import seaborn as sns
import os, sys
import datetime
import numpy as np
import pandas as pd

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
import pymc as pm

In [ ]:
az.style.use("seaborn")
datetime.datetime.now()

In [ ]:
RANDOM_SEED = 749195
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(2)
sys.stderr = open(os.devnull, "w")

In [ ]:
data = pd.read_csv("./data/databricks_cost_data/explorer-cost-graph-2023-02-08.csv", index_col=0).T
data.columns.name = "cluster_name"
data.index = pd.to_datetime(data.index)
data.index.name = "date"
unstacked_data = data.stack().reset_index()
unstacked_data.columns = ["date", "cluster_name", "cost"]

In [ ]:
cost_per_cluster_type = unstacked_data.assign(is_single_user=unstacked_data.cluster_name.str.contains("su_cluster"))\
    .pivot_table(index="date", columns="is_single_user", values="cost", aggfunc="sum")
cost_per_cluster_type.columns = ["bu_cluster", "su_cluster"]


In [ ]:
fig, ax = plt.subplots(figsize=(30, 5))

cost_per_cluster_type.plot.bar(width=0.75, stacked=True, ax=ax)

In [ ]:
coords = {
    "date_ids": np.arange(cost_per_cluster_type.shape[0]),
    "date": data.index.values,
    "users": unstacked_data.cluster_name.drop_duplicates().str.extract("(.*)_su_cluster_spark3").dropna().iloc[:, 0].unique(),
    "bu_cluster": "axp-bu-cluster"
}

coords["su_clusters"] = [f"{c}_su_cluster_spark3" for c in coords["users"]]

In [ ]:
with pm.Model(coords=coords):
    usage_sigma = pm.InverseGamma("usage_sigma", alpha=2, beta=1)
    usage_level = pm.HalfNormal("usage_level", sigma=usage_sigma, dims=["date_ids", "users"])
    adoption_swithpoint = pm.DiscreteUniform("adoption_swithpoint", lower=coords["date_ids"][0], upper=coords["date_ids"][-1])
    early_bu_cluster_adoption = pm.Beta("early_bu_cluster_adoption", alpha=1, beta=2)
    later_bu_cluster_adoption = pm.Beta("later_bu_cluster_adoption", alpha=2, beta=1)
    
    dims = pm.math.ones_like(usage_level)
    dates_dims = np.repeat(coords["date_ids"][:,None], len(coords["users"]), axis=1)
    bu_cluster_adoption = pm.math.switch(adoption_swithpoint * dims >= dates_dims, early_bu_cluster_adoption, later_bu_cluster_adoption)
    
    su_cluster_daily_cost_per_usage = pm.Pareto("su_cluster_daily_cost_per_usage", m=1, alpha=1)
    bu_cluster_daily_cost_per_usage = pm.Pareto("bu_cluster_daily_cost_per_usage", m=1, alpha=1)
    
    user_cluster_cost = pm.Normal(
        "user_cluster_cost",
        mu=su_cluster_daily_cost_per_usage * usage_level * (1 - bu_cluster_adoption), 
        sigma=1,
        observed=data.loc[:, coords["su_clusters"]].values
    ) 
    
    bu_cluster_cost = pm.Normal(
        "bu_cluster_cost",
        mu=bu_cluster_daily_cost_per_usage * (usage_level * bu_cluster_adoption).sum(axis=1),
        sigma=1,
        observed=data.loc[:, coords["bu_cluster"]].values
    
    )
    
    simulated_usage = pm.HalfNormal("simulated_usage", sigma=usage_sigma, dims=["users"])
    simulated_su_cluster_cost = pm.Normal(
        "simulated_su_cluster_cost",
        mu=su_cluster_daily_cost_per_usage * simulated_usage * 0.1,
        sigma=1
    )
    
    simulated_bu_cluster_cost = pm.Normal(
        "simulated_bu_cluster_cost",
        mu=(bu_cluster_daily_cost_per_usage * simulated_usage * 0.9).sum(),
        sigma=1
    )
    
    trace = pm.sample(target_accept=0.9, draws=1000, discard_tuned_samples=True, tune=200)

In [ ]:
az.plot_posterior(trace, var_names=["adoption_swithpoint", "early_bu_cluster_adoption", "later_bu_cluster_adoption"])

In [ ]:
az.plot_posterior(trace, var_names=["simulated_su_cluster_cost", "simulated_bu_cluster_cost"])

In [ ]:
az.plot_posterior(trace, var_names=["bu_cluster_daily_cost_per_usage", "su_cluster_daily_cost_per_usage"])

In [ ]:
usage_level = trace.posterior.usage_level.to_dataframe().unstack(level=3).unstack(level=2).usage_level
usage_level.head()

In [ ]:
fig, ax = plt.subplots(ncols=2)
usage_level.thais_almeida.mean().plot(ax=ax[0])
data.thais_almeida_su_cluster_spark3.plot(ax=ax[1])

In [ ]:
simulated_bu_cluster_cost = trace.posterior.simulated_bu_cluster_cost.to_dataframe().simulated_bu_cluster_cost
simulated_bu_cluster_cost

In [ ]:
total_simulated_su_cluster_cost = trace.posterior.simulated_su_cluster_cost.to_dataframe().unstack().sum(axis=1)
total_simulated_su_cluster_cost

In [ ]:
simulated_total_cost_post_bu_cluster = simulated_bu_cluster_cost + total_simulated_su_cluster_cost
simulated_total_cost_post_bu_cluster

In [ ]:
simulated_total_cost_pre_bu_cluster = total_simulated_su_cluster_cost / 0.1 # not using BU cluster
simulated_total_cost_pre_bu_cluster

In [ ]:
ratio = simulated_total_cost_post_bu_cluster / simulated_total_cost_pre_bu_cluster
ratio.hist(bins=np.linspace(0, 2, 101))

In [ ]:
az.plot_posterior(ratio.loc[(ratio > 0) & (ratio < 2)].values)